<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/TCGA-1_Classical_Stats_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
# Upload Files (.csv and R data)
from google.colab import files
import pandas as pd
import numpy as np
from functools import reduce
from IPython.display import display

#uploaded = files.upload()

In [51]:
# Cell 1: Setup, Data Loading & Harmonization
import pandas as pd
import numpy as np
from IPython.display import display

# Display settings for Pandas
pd.set_option('display.max_columns', 50)

# We will standardize all patient ID columns to this name
ID_COLUMN = 'Patient_ID'

def load_omics_csv(filepath):
    """Loads Omics data, transposes it, and standardizes the TCGA Patient IDs."""
    try:
        # Read CSV and transpose so rows = samples, columns = features
        df = pd.read_csv(filepath, index_col=0).T

        # Convert the index into a standard column
        df = df.reset_index().rename(columns={'index': ID_COLUMN})

        # HARMONIZE IDs: Replace dots with dashes and keep only the first 12 characters (e.g., 'TCGA-A1-A0SH')
        df[ID_COLUMN] = df[ID_COLUMN].str.replace('.', '-', regex=False).str[0:12]

        # Drop duplicates just in case multiple samples exist for the same patient
        df = df.drop_duplicates(subset=[ID_COLUMN])
        return df
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return pd.DataFrame()

# 1. Load clinical data
print("Loading datasets...")
clinical_df = pd.read_csv('Table1Nature.csv')

# Standardize the clinical ID column name and format
if 'Complete TCGA ID' in clinical_df.columns:
    clinical_df = clinical_df.rename(columns={'Complete TCGA ID': ID_COLUMN})
clinical_df[ID_COLUMN] = clinical_df[ID_COLUMN].astype(str).str[0:12]

print(f"Clinical Data Shape: {clinical_df.shape}")

# 2. Load Omics data
ge_df = load_omics_csv('context1_GE.csv')
meth_df = load_omics_csv('context2_Meth.csv')
mirna_df = load_omics_csv('context3_miRNA.csv')
prot_df = load_omics_csv('context4_Protein.csv')

print(f"Gene Expression Shape: {ge_df.shape}")
print(f"Methylation Shape: {meth_df.shape}")
print(f"miRNA Shape: {mirna_df.shape}")
print(f"Protein Shape: {prot_df.shape}")

Loading datasets...
Clinical Data Shape: (825, 30)
Gene Expression Shape: (348, 646)
Methylation Shape: (348, 575)
miRNA Shape: (348, 424)
Protein Shape: (348, 172)


In [52]:
# Cell 2: Clinical Data Summaries Using Pandas
print("\n--- Numeric Summaries ---")
display(clinical_df.describe().round(2))

print("\n--- Categorical Distributions ---")
categorical_cols = ['Gender', 'ER Status', 'PR Status', 'HER2 Final Status', 'PAM50 mRNA']

for col in categorical_cols:
    if col in clinical_df.columns:
        counts = clinical_df[col].value_counts(dropna=False)
        pcts = clinical_df[col].value_counts(dropna=False, normalize=True) * 100
        summary_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
        print(f"\n{col}:")
        display(summary_df)

print("\n--- Missing Data Analysis ---")
missing = clinical_df.isnull().sum()
missing_pct = (missing / len(clinical_df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct (%)': missing_pct})
display(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Pct (%)', ascending=False))


--- Numeric Summaries ---


,Age at Initial Pathologic Diagnosis,Days to date of Death,OS event,OS Time,SigClust Unsupervised mRNA,SigClust Intrinsic mRNA,miRNA Clusters,methylation Clusters,CN Clusters,Integrated Clusters (with PAM50),Integrated Clusters (no exp),Integrated Clusters (unsup exp)
count,818.00,93.00,818.00,818.00,522.00,522.00,694.00,799.00,773.00,348.00,348.00,348.00
mean,58.04,1743.75,0.11,912.26,-4.79,-6.76,4.18,3.11,2.71,2.79,2.20,2.64
std,13.21,1143.74,0.32,1072.09,3.34,4.81,1.66,1.39,1.33,0.91,1.01,1.00
min,26.00,157.00,0.00,0.00,-12.00,-13.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,48.00,811.00,0.00,178.00,-7.00,-12.00,3.00,2.00,2.00,2.00,1.00,2.00
50%,58.00,1563.00,0.00,544.00,-3.00,-5.00,4.00,3.00,3.00,3.00,2.00,3.00
75%,67.00,2520.00,0.00,1293.25,-3.00,-2.00,6.00,4.00,4.00,3.00,3.00,3.00
max,90.00,4456.00,1.00,7125.00,0.00,0.00,7.00,5.00,5.00,4.00,5.00,5.00



--- Categorical Distributions ---

Gender:


,Count,Percentage (%)
Gender,,
FEMALE,810,98.18
MALE,8,0.97
NaN,7,0.85



ER Status:


...

,Count,Percentage (%)
HER2 Final Status,,
Negative,652,79.03
Positive,114,13.82
NaN,34,4.12
Not Available,15,1.82
Equivocal,10,1.21



PAM50 mRNA:


,Count,Percentage (%)
PAM50 mRNA,,
NaN,303,36.73
Luminal A,231,28.00
Luminal B,127,15.39
Basal-like,98,11.88
HER2-enriched,58,7.03
Normal-like,8,0.97



--- Missing Data Analysis ---


,Missing_Count,Missing_Pct (%)
Days to date of Death,732,88.727273
Integrated Clusters (no exp),477,57.818182
Integrated Clusters (unsup exp),477,57.818182
Integrated Clusters (with PAM50),477,57.818182
RPPA Clusters,422,51.151515
SigClust Intrinsic mRNA,303,36.727273
PAM50 mRNA,303,36.727273
SigClust Unsupervised mRNA,303,36.727273
miRNA Clusters,131,15.878788
CN Clusters,52,6.303030


In [53]:
# Cell 3: Global Statistics for Gene Expression
# Drop the ID column temporarily just for calculating numeric stats
if ID_COLUMN in ge_df.columns:
    numeric_ge_df = ge_df.drop(columns=[ID_COLUMN])
else:
    numeric_ge_df = ge_df

omics_summary = pd.DataFrame({
    'Mean': numeric_ge_df.mean(axis=0),
    'Variance': numeric_ge_df.var(axis=0),
    'Min': numeric_ge_df.min(axis=0),
    'Max': numeric_ge_df.max(axis=0),
    'Median': numeric_ge_df.median(axis=0)
})

print("\nTop 10 Most Variable Genes:")
display(omics_summary.sort_values(by='Variance', ascending=False).head(10))

# Filter dataset to keep only the top 1000 most variable genes
top_1000_genes = omics_summary.nlargest(1000, 'Variance').index.tolist()
print(f"\nIdentified {len(top_1000_genes)} high-variance genes.")


Top 10 Most Variable Genes:


,Mean,Variance,Min,Max,Median
447,-1.722206,12.251936,-10.168250,2.835250,0.0
276,0.181373,10.486483,-6.251000,8.578000,0.0
593,-0.176296,10.247764,-7.138250,5.919250,0.0
329,-0.461939,9.868949,-7.384625,5.553375,0.0
413,-0.211610,9.566383,-7.683000,6.123600,0.0
331,-0.786924,9.010733,-7.529661,4.109250,0.0
115,1.329759,7.844445,-2.951625,6.674125,0.0
365,1.262632,7.597922,-2.613000,9.817000,0.0
423,-0.917668,7.526465,-7.962958,3.008542,0.0
611,0.794482,7.523447,-2.481250,10.640625,0.0



Identified 645 high-variance genes.


In [54]:
# Cell 4: MERGE DATASETS
# 1. Put all loaded DataFrames into a list
dfs_to_merge = [clinical_df, ge_df, meth_df, mirna_df, prot_df]

# 2. Filter out any empty dataframes
dfs_to_merge = [df for df in dfs_to_merge if not df.empty]

# 3. Initialize merged_df with the first item (clinical_df)
merged_df = dfs_to_merge[0]

# 4. Loop through the remaining dataframes and merge iteratively
for df in dfs_to_merge[1:]:
    merged_df = pd.merge(merged_df, df, on=ID_COLUMN, how='inner')

print(f"\nMerged Master Dataset Shape: {merged_df.shape}")


Merged Master Dataset Shape: (348, 1843)


In [55]:
# Cell 5: Differential Expression (ER+ vs ER-)
if 'ER Status' in merged_df.columns:
    analysis_df = merged_df[merged_df['ER Status'].isin(['Positive', 'Negative'])]

    # Ensure we only check columns that exist in the merged dataframe
    gene_cols = [col for col in top_1000_genes if col in analysis_df.columns]

    grouped = analysis_df.groupby('ER Status')[gene_cols]
    n = grouped.count().astype(float)
    means = grouped.mean()
    variances = grouped.var()

    mean_pos, mean_neg = means.loc['Positive'], means.loc['Negative']
    var_pos, var_neg = variances.loc['Positive'], variances.loc['Negative']
    n_pos, n_neg = n.loc['Positive'], n.loc['Negative']

    mean_diff = mean_pos - mean_neg
    standard_error = np.sqrt((var_pos / n_pos) + (var_neg / n_neg))
    t_stat = mean_diff / standard_error

    stat_results = pd.DataFrame({
        'Mean_ER_Pos': mean_pos,
        'Mean_ER_Neg': mean_neg,
        'Mean_Difference': mean_diff,
        'T_Statistic': t_stat,
        'Abs_T_Statistic': np.abs(t_stat)
    }).sort_values(by='Abs_T_Statistic', ascending=False)

    print("\nTop 10 Differentially Expressed Genes (ER+ vs ER-):")
    display(stat_results.head(10).round(4))
else:
    print("\nError: 'ER Status' column not found for Differential Expression analysis.")


Top 10 Differentially Expressed Genes (ER+ vs ER-):


,Mean_ER_Pos,Mean_ER_Neg,Mean_Difference,T_Statistic,Abs_T_Statistic
47,0.5063,-1.3407,1.8470,23.9588,23.9588
447,-0.1879,-6.4357,6.2478,18.9888,18.9888
544,0.2825,-2.5750,2.8575,18.4472,18.4472
48,0.4678,-1.1607,1.6285,17.2191,17.2191
603,0.2099,-3.4171,3.6270,16.5391,16.5391
645,0.7603,-1.7297,2.4900,16.1059,16.1059
563,0.2580,-3.0383,3.2963,15.2768,15.2768
539,-0.0198,-3.9532,3.9334,14.3800,14.3800
591,0.8987,-1.2888,2.1875,14.3587,14.3587
540,0.1591,-2.7055,2.8646,14.3245,14.3245


In [56]:
# Cell 6: Feature Correlation Analysis
if 'stat_results' in locals() and not stat_results.empty:
    top_15_genes = stat_results.head(15).index.tolist()

    corr_matrix = merged_df[top_15_genes].corr(method='pearson')

    print("\nCorrelation Matrix of Top 15 Differentially Expressed Genes:")
    display(corr_matrix.style.background_gradient(cmap='RdBu_r', axis=None, vmin=-1, vmax=1).format("{:.2f}"))
else:
    print("\nCannot calculate correlation matrix because stat_results is empty.")


Correlation Matrix of Top 15 Differentially Expressed Genes:


,47,447,544,48,603,645,563,539,591,540,448,643,592,565,579
47,1.00,0.75,0.76,0.82,0.65,0.67,0.67,0.66,0.38,0.68,0.63,-0.65,0.63,0.57,0.59
447,0.75,1.00,0.77,0.68,0.69,0.66,0.77,0.74,0.50,0.74,0.67,-0.68,0.65,0.53,0.58
544,0.76,0.77,1.00,0.66,0.62,0.66,0.72,0.67,0.40,0.70,0.59,-0.68,0.68,0.56,0.47
48,0.82,0.68,0.66,1.00,0.55,0.60,0.57,0.64,0.43,0.67,0.62,-0.62,0.51,0.49,0.48
603,0.65,0.69,0.62,0.55,1.00,0.56,0.63,0.57,0.47,0.60,0.47,-0.58,0.54,0.48,0.51
645,0.67,0.66,0.66,0.60,0.56,1.00,0.65,0.50,0.42,0.62,0.60,-0.59,0.52,0.43,0.42
563,0.67,0.77,0.72,0.57,0.63,0.65,1.00,0.66,0.38,0.67,0.57,-0.60,0.65,0.60,0.51
539,0.66,0.74,0.67,0.64,0.57,0.50,0.66,1.00,0.32,0.66,0.60,-0.58,0.56,0.53,0.66
591,0.38,0.50,0.40,0.43,0.47,0.42,0.38,0.32,1.00,0.44,0.38,-0.48,0.49,0.33,0.22
540,0.68,0.74,0.70,0.67,0.60,0.62,0.67,0.66,0.44,1.00,0.58,-0.69,0.63,0.53,0.54
